## Análise Básica

**_Resposta a Objetivo Principal_**

Análise de dados de notificações em série temporal por categoria NTEP (incluindo arboviroses que estão fora do Decreto)

In [0]:
%sql
SELECT d.ANO_NOTIF, d.CATEGORIA_NTEP, d.PRESENTE_NO_NTEP,
       SUM(f.N_NOTIFICACOES) AS total_notificacoes
FROM workspace.sinan.fato_notificacoes_ntep f
JOIN workspace.sinan.dim_notificacao d ON f.SK_NOTIFICACAO = d.SK_NOTIFICACAO
GROUP BY d.ANO_NOTIF, d.CATEGORIA_NTEP, d.PRESENTE_NO_NTEP
ORDER BY total_notificacoes DESC;

**_Resposta a Objetivo Secundário 1_**

Zika e Chikungunya figuram ao menos 7x entre as top 25 categorias mais notificadas na análise da série histórica, o que aponta para a sensibilidade do Decreto do NTEP ser atualizado para incluir agravos de arboviroses além da Dengue. 


Notificações por sexo e raça

In [0]:
%sql
SELECT n.SEXO, n.RACA, SUM(f.N_NOTIFICACOES) AS total_notificacoes
FROM workspace.sinan.fato_notificacoes_ntep f
JOIN workspace.sinan.dim_notificante n ON f.SK_NOTIFICANTE = n.SK_NOTIFICANTE
GROUP BY n.SEXO, n.RACA
ORDER BY total_notificacoes DESC;

Notificações por região e UF

In [0]:
%sql
SELECT l.REGIAO_NOTIF, SUM(f.N_NOTIFICACOES) AS total_notificacoes
FROM workspace.sinan.fato_notificacoes_ntep f
JOIN workspace.sinan.dim_local_notificacao l ON f.SK_LOCAL = l.SK_LOCAL
GROUP BY l.REGIAO_NOTIF
ORDER BY total_notificacoes DESC;

In [0]:
%sql
SELECT l.UF_NOTIF, SUM(f.N_NOTIFICACOES) AS total_notificacoes
FROM workspace.sinan.fato_notificacoes_ntep f
JOIN workspace.sinan.dim_local_notificacao l ON f.SK_LOCAL = l.SK_LOCAL
GROUP BY l.UF_NOTIF
ORDER BY total_notificacoes DESC;

10 categorias do NTEP mais notificadas, por UF

In [0]:
%sql
WITH base AS (
  SELECT d.CATEGORIA_NTEP,
         SUM(f.N_NOTIFICACOES) AS total_notificacoes
  FROM workspace.sinan.fato_notificacoes_ntep f
  JOIN workspace.sinan.dim_notificacao d ON f.SK_NOTIFICACAO = d.SK_NOTIFICACAO
  WHERE d.PRESENTE_NO_NTEP = true
  GROUP BY d.CATEGORIA_NTEP
)
SELECT *,
       RANK() OVER (ORDER BY total_notificacoes DESC) AS rank_categoria_na_uf
FROM base
QUALIFY rank_categoria_na_uf <= 10
ORDER BY rank_categoria_na_uf;

3 categorias do NTEP, Raça e Sexo mais notificadas

In [0]:
%sql
WITH base AS (
  SELECT n.SEXO, n.RACA, d.CATEGORIA_NTEP,
         SUM(f.N_NOTIFICACOES) AS total_notificacoes
  FROM workspace.sinan.fato_notificacoes_ntep f
  JOIN workspace.sinan.dim_notificante n ON f.SK_NOTIFICANTE = n.SK_NOTIFICANTE
  JOIN workspace.sinan.dim_notificacao d ON f.SK_NOTIFICACAO = d.SK_NOTIFICACAO
  WHERE d.PRESENTE_NO_NTEP = true
  GROUP BY n.SEXO, n.RACA, d.CATEGORIA_NTEP
)
SELECT *,
       RANK() OVER (ORDER BY total_notificacoes DESC) AS rank_categoria_na_uf
FROM base
QUALIFY rank_categoria_na_uf <= 3
ORDER BY rank_categoria_na_uf;

Categorias mais notificadas por grupos populacionais específicos

In [0]:
%sql
WITH base AS (
  SELECT n.SEXO, n.RACA, d.CATEGORIA_NTEP, d.PRESENTE_NO_NTEP,
         SUM(f.N_NOTIFICACOES) AS total_notificacoes
  FROM workspace.sinan.fato_notificacoes_ntep f
  JOIN workspace.sinan.dim_notificante n ON f.SK_NOTIFICANTE = n.SK_NOTIFICANTE
  JOIN workspace.sinan.dim_notificacao d ON f.SK_NOTIFICACAO = d.SK_NOTIFICACAO
  WHERE (n.SEXO = 'Feminino' AND n.RACA = 'Preta')
     OR (n.SEXO = 'Masculino' AND n.RACA = 'Branca')
  GROUP BY n.SEXO, n.RACA, d.CATEGORIA_NTEP, d.PRESENTE_NO_NTEP
)
SELECT *,
       RANK() OVER (PARTITION BY SEXO, RACA ORDER BY total_notificacoes DESC) AS rank_grupo
FROM base
QUALIFY rank_grupo <= 5
ORDER BY SEXO, RACA, rank_grupo;

### Autoavaliação
O objetivo de aprendizado foi cumprido. 
A coleta e processamento de +39M de registros do FTP do SINAN de interesse para o seguro social acidentário foi possibilitada por meio de scripts que usam pyspark e análise sql e download paralelizado, com dados já categorizados (pulou-se a camada Bronze). 
Esses números podem ser confrontados com dados abertos de Comunicação de Acidente de Trabalho (https://dados.gov.br/dados/conjuntos-dados/inss-comunicacao-de-acidente-de-trabalho-cat1). Esses números também podem ser enriquecidos com dados adicionais (ex. nome dos municípios de interesse, classificação de IDH, entre outros) para uma análise mais aprofundada do retrato de notificações obrigatórias do SINAN, visando informar atualização do Decreto do NTEP ou direcionais esforços de monitoramento e coordenação entre notificações feitas no sistema de saúde e outras diretamente pelos empregadores à Previdência Social.